In [ ]:
# Run once
%pip install pgmpy kaggle pandas scikit-learn matplotlib seaborn pyarrow

  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
  Using cached jsonschema_specifications-2025.9.1-py3-none-any.whl.metadata (2.9 kB)
  Using cached referencing-0.37.0-py3-none-any.whl.metadata (2.8 kB)
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ------------- -------------------------- 0.8/2.4 MB 4.8 MB/s eta 0:00:01
   ------------------------------ --------- 1.8/2.4 MB 4.7 MB/s eta 0:00:01
   ---------------------------------------- 2.4/2.4 MB 4.5 MB/s  0:00:00
   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   --- ----------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress t

In [ ]:
# run once, downloads 2002 and 2003 GSOD data via HTTP (parallel)
import requests
import os
import re
from concurrent.futures import ThreadPoolExecutor, as_completed

os.makedirs("../data/raw/gsod", exist_ok=True)

BASE_URL = "https://www.ncei.noaa.gov/data/global-summary-of-the-day/access/"

def download_station(station, year_url, year):
    """Download a single station file (skip if already exists)."""
    filepath = f"../data/raw/gsod/{year}/{station}"
    
    # Skip if already downloaded
    if os.path.exists(filepath):
        return None  # Already exists
    
    try:
        response = requests.get(f"{year_url}{station}", timeout=30)
        response.raise_for_status()
        with open(filepath, 'wb') as f:
            f.write(response.content)
        return True
    except Exception as e:
        return False

def download_year_fast(year, max_workers=50):
    """Download GSOD data for a year using parallel requests."""
    year_url = f"{BASE_URL}{year}/"
    print(f"Fetching: {year_url}")
    
    # Get list of station files
    r = requests.get(year_url, timeout=60)
    r.raise_for_status()
    
    stations = re.findall(r'href="(\d{11}\.csv)"', r.text)
    print(f"Found {len(stations)} station files for {year}")
    
    os.makedirs(f"../data/raw/gsod/{year}", exist_ok=True)
    
    # Filter out already-downloaded stations
    existing = set(os.listdir(f"../data/raw/gsod/{year}"))
    to_download = [s for s in stations if s not in existing]
    print(f"Already have {len(existing)} files, need to download {len(to_download)}")
    
    if not to_download:
        print(f"Year {year} already complete!")
        return len(existing)
    
    # Download in parallel (only new files)
    success_count = 0
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(download_station, s, year_url, year): s for s in to_download}
        
        for i, future in enumerate(as_completed(futures)):
            result = future.result()
            if result is True:
                success_count += 1
            if (i + 1) % 2000 == 0:
                print(f"  Progress: {i+1}/{len(to_download)}")
    
    total = len(existing) + success_count
    print(f"Downloaded {success_count}/{len(to_download)} new files (total: {total}/{len(stations)})")
    return success_count

# Download both years in parallel
print("=" * 50)
print("Downloading 2002 GSOD data (parallel)...")
download_year_fast(2002)

print("=" * 50)
print("Downloading 2003 GSOD data (parallel)...")
download_year_fast(2003)

print("Done!")

Fetching: https://www.ncei.noaa.gov/data/global-summary-of-the-day/access/2002/
Found 8990 station files for 2002
  Progress: 2000/8990
  Progress: 4000/8990


In [ ]:
# Read the CSV files into a single DataFrame
import pandas as pd
import glob

COL_NAMES = [
    "STN", "WBAN", "YEARMODA",
    "TEMP", "TEMP_CNT", "DEWP", "DEWP_CNT",
    "SLP",  "SLP_CNT",  "STP",  "STP_CNT",
    "VISIB","VISIB_CNT","WDSP", "WDSP_CNT",
    "MXSPD","GUST","MAX","MIN","PRCP","SNDP","FRSHTT"
]

# NOAA missing value sentinels per element
SENTINELS = {
    "TEMP": 9999.9, "DEWP": 9999.9, "SLP": 9999.9, "STP": 9999.9,
    "VISIB": 999.9, "WDSP": 999.9,  "MXSPD": 999.9, "GUST": 9999.9,
    "MAX": 9999.9,  "MIN": 9999.9,  "PRCP": 99.99,   "SNDP": 999.9,
}

def read_gsod_csv(filepath):
    df = pd.read_csv(filepath, sep=r"\s+", header=None, names=COL_NAMES)
    
    # MAX and MIN may carry a trailing '*' flag indicating a provisional value
    df["MAX"] = pd.to_numeric(
        df["MAX"].astype(str).str.replace("*", "", regex=False), errors="coerce"
    )
    df["MIN"] = pd.to_numeric(
        df["MIN"].astype(str).str.replace("*", "", regex=False), errors="coerce"
    )
    # PRCP carries a trailing letter flag (A–I indicating accumulation window)
    df["PRCP"] = pd.to_numeric(
        df["PRCP"].astype(str).str.replace(r"[A-Za-z]", "", regex=True), errors="coerce"
    )
    
    # Apply missing sentinels
    for col, sentinel in SENTINELS.items():
        df[col] = df[col].replace(sentinel, pd.NA)
    
    return df

for year in [2002, 2003]:
    csv_files = glob.glob(f"../data/raw/gsod/{year}/*.csv")
    print(f"Found {len(csv_files)} station files for {year}")
    
    df_raw = pd.concat([read_gsod_csv(f) for f in csv_files], ignore_index=True)
    print(f"Total records for {year}: {len(df_raw):,}")
    df_raw.to_csv(f"../data/processed/gsod_{year}.csv", index=False)
    print(f"Saved → ../data/processed/gsod_{year}.csv")

FileNotFoundError: [Errno 2] No such file or directory: '../data/raw/gsod/gsod_2002.tar'

In [ ]:
# Parse YEARMODA → proper date
df_raw["DATE"] = pd.to_datetime(df_raw["YEARMODA"].astype(str), format="%Y%m%d")

# FRSHTT indicates the following Fog | Rain | Snow | Hail | Thunder | Tornado
df_raw["FRSHTT"]  = df_raw["FRSHTT"].astype(str).str.zfill(6)
df_raw["FOG"]     = df_raw["FRSHTT"].str[0].astype(int)
df_raw["RAIN"]    = df_raw["FRSHTT"].str[1].astype(int)
df_raw["SNOW"]    = df_raw["FRSHTT"].str[2].astype(int)
df_raw["HAIL"]    = df_raw["FRSHTT"].str[3].astype(int)
df_raw["THUNDER"] = df_raw["FRSHTT"].str[4].astype(int)
df_raw["TORNADO"] = df_raw["FRSHTT"].str[5].astype(int)

print(df_raw.dtypes)
print(df_raw[["DATE","TEMP","DEWP","SLP","VISIB","WDSP","MXSPD","GUST",
               "MAX","MIN","PRCP","SNDP","RAIN","SNOW","FOG"]].describe())

In [ ]:
# Save the cleaned DataFrame to CSV
# Need to check for handling missing vals and some more cleaning, some format changes maybe
# Need to check more on what Bayes networks want for data format and input types
# have not checked outliers yet either
# need station filtering maybe? depends on the route we want to go.

os.makedirs("../data/processed", exist_ok=True)
df_raw.to_csv("../data/processed/gsod_2003.csv", index=False)
print("Saved → ../data/processed/gsod_2003.csv")

**Notes from meeting

need a network structure

- pressure and temperature are more "fundamental" factors
- precipitation will be rain and snow, temp and pressure related
- wind will be pressure and temperatre impacted, not by precip
- dewpoint - idk, need to learn more weather
- fog plays in ?????
- for Bayes there should be binning for each type but need classifiers?
- pressure = high/low
- temp = cold, avg, warm
- wind = none, low, high ???
- dew point ??
- fog, rain, snow can either be bools or maybe light/heavy/none classifications?    For the precipitations have precip be light/heavy then can just be bools for the type of precip? fog is still ??
- 

Meeting Notes:
double check for provided bayesian graph information, otherwise look into other sources for graph templates. 
If we can't find the information ask the TA for assistance

take a look at a simplified dynamic bayesian network through another model. using t-1 to see hsitory, this may simplify our process.
^ The hidden markov model is what may be good here.